# Import and Install Dependencies

In [10]:
import os
import cv2
import numpy as np
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.utils import to_categorical
from keras.callbacks import TensorBoard
from sklearn.model_selection import train_test_split
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

## draw style of landmarks like color, thickness and circle_radius

In [11]:
# Setup Folders for Collection
actions = np.array(['weekend','saturday','sunday','monday','tuesday','wednesday','thursday','friday']) # Actions that we try to detect
no_sequences = 500  # Thirty videos worth of data
sequence_length = 15  # Videos are going to be 30 frames in length

In [12]:
# Preprocess Data and Create Labels and Features
label_map = {label:num for num, label in enumerate(actions)} # create labels for actions like this {'hello': 0, 'thanks': 1, 'iloveyou': 2}

# array sequences: should be a big array containing all videos of all actions and this is features data or X data  
# array labels: should be a big array containing all labels of all actions and this is labels data or y data  
# Initialize an empty list to store the file paths
# Initialize an empty list to store the data

sequences, labels = [], []
for action in actions:
    for sequence in range(no_sequences):

        res = np.load(os.path.join("/kaggle/input/weekdays/weekDays", action, "{}.npy".format(sequence)))

        # Reshape res from (30, 543, 3) to (30, 1629)
        res = res.reshape(res.shape[0], -1)    
        sequences.append(res)
        labels.append(label_map[action])

X = np.array(sequences) # sequences ---> (720, 30, 1629)
y = to_categorical(labels).astype(int) # labels --> (720, 2)

In [13]:
X.shape

(3200, 10, 1629)

In [14]:
# X_train --> (228, 30, 1629)
# y_train --> (228, 2)
# X_test --> (12, 30, 1629)
# y_test --> (12, 2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05) # 95% train and 5% test

In [15]:
actions.shape[0]

8

In [33]:
# Build and Train LSTM Neural Network
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(15,1629)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model.fit(X_train, y_train, epochs=1000)
model.summary()

Epoch 1/200
95/95 [==============================] - 4s 15ms/step - loss: 1.7704 - categorical_accuracy: 0.2296
Epoch 2/200
95/95 [==============================] - 1s 15ms/step - loss: 1.5791 - categorical_accuracy: 0.2803
Epoch 3/200
95/95 [==============================] - 1s 14ms/step - loss: 1.3676 - categorical_accuracy: 0.4138
Epoch 4/200
95/95 [==============================] - 1s 15ms/step - loss: 1.2791 - categorical_accuracy: 0.4477
Epoch 5/200
95/95 [==============================] - 1s 14ms/step - loss: 1.1750 - categorical_accuracy: 0.5211
Epoch 6/200
95/95 [==============================] - 1s 15ms/step - loss: 1.0623 - categorical_accuracy: 0.5661
Epoch 7/200
95/95 [==============================] - 1s 15ms/step - loss: 0.9618 - categorical_accuracy: 0.6095
Epoch 8/200
95/95 [==============================] - 1s 15ms/step - loss: 0.9914 - categorical_accuracy: 0.6049
Epoch 9/200
95/95 [==============================] - 1s 14ms/step - loss: 0.8209 - categorical_accuracy:

In [34]:
# Evaluation using Confusion Matrix and Accuracy
yhat = model.predict(X_test)

ytrue = np.argmax(y_test, axis=1).tolist()
yhat = np.argmax(yhat, axis=1).tolist()

print(accuracy_score(ytrue, yhat))

5/5 [==============================] - 0s 7ms/step
0.95625


In [19]:
# save weights
model.save('weekDays3.h5')

/opt/conda/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
